# DRL Walkforward Training Pipeline for Colab

This notebook is designed to run the Deep Reinforcement Learning (DQN) walkforward training on Google Colab. By running this on Colab, you can leverage free GPUs (or TPUs) to drastically speed up training, while persisting the trained models directly back to your Google Drive.

## 1. Mount Google Drive
Mounting Google Drive allows us to access the project codebase and save the resulting model checkpoints (`drl/dqn/models/`) and logs so they are not lost when the Colab session disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Setup Workspace

**Pre-requisite:** You need to have your `IEOR4733_Project` repository inside your Google Drive. 
You can either:
1. Upload the entire local folder to Google Drive (e.g., to `My Drive/IEOR4733_Project`).
2. Zip it locally, upload the zip, and use `!unzip` here.
3. Or `!git clone` if your repository is pushed to GitHub.

Below we assume the project is directly in your Google Drive.

In [ ]:
import os

# Update this path if you placed the project somewhere else in your Drive
PROJECT_DIR = '/content/drive/MyDrive/IEOR4733_Project'

if not os.path.exists(PROJECT_DIR):
    print(f"Directory {PROJECT_DIR} not found. Please upload or clone the repo there.")
else:
    os.chdir(PROJECT_DIR)
    print(f"Successfully changed working directory to: {os.getcwd()}")
    
    # List files to verify we are in the right place
    !ls -la

## 3. Install Dependencies
Google Colab already comes with PyTorch, Pandas, Numpy, and Matplotlib pre-installed. We only need to install any extra packages if they are missing.

In [ ]:
# If you have a requirements.txt, you can uncomment the next line:
# !pip install -r requirements.txt

# Let's verify we have GPU access
!nvidia-smi

## 4. Feature Preparation
Prepare the shared feature artifacts. This step ensures that the `train_start_idx`, `test_start_idx`, etc., are computed safely and stored as `.pkl` artifacts, maintaining strict walkforward isolation.

In [ ]:
# We run it for Forex as an example. You can change this to Equities or Commodities.
!python -m drl_shared.prepare_features --asset Forex

## 5. Feature Data Sanity Check
Ensure that the generated `.npz` feature artifacts are clean (no NaNs, no Infs, not all-zeros, and no excessive duplicate rows).

In [ ]:
!python -m drl_shared.sanity_check_features --asset Forex --round 1

## 6. Multi-Seed Walkforward Training
Start the multi-seed DQN training process. This script utilizes multi-processing to train multiple seeds concurrently.

> **Note:** Make sure your Colab Runtime has a GPU enabled (`Runtime > Change runtime type > Hardware accelerator > T4 GPU`).

In [ ]:
!python drl/dqn/train/train_walkforward_multiseed.py \
    --asset Forex \
    --seeds 5 \
    --episodes 200 \
    --device cuda

## 7. Verification and Backtest
Run a quick test to verify the shared DQN logic and boundaries for the newly trained model.

In [ ]:
!python drl/dqn/tests/verify_shared_dqn.py --asset Forex --round 1 --ticker AN --require-prepared